# ADHD Tools Colab Notebook

Run each cell to interact with simplified versions of the tools. This notebook stores data in a folder named **ADHDtools** inside your Google Drive.


In [ ]:
from google.colab import drive
import os, json

drive.mount('/content/drive')
DATA_DIR = '/content/drive/My Drive/ADHDtools'
os.makedirs(DATA_DIR, exist_ok=True)

def load_data(name, default=None):
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return default if default is not None else {}

def save_data(name, data):
    with open(os.path.join(DATA_DIR, name), 'w') as f:
        json.dump(data, f)


## Pomodoro Timer

In [ ]:
import time, threading
import ipywidgets as widgets
from IPython.display import display

pomo_minutes = widgets.IntText(value=25, description='Minutes:')
start_btn = widgets.Button(description='Start')
stop_btn = widgets.Button(description='Stop')
status = widgets.Label(value='Ready')
display(pomo_minutes, start_btn, stop_btn, status)

stop_flag = False

def run_pomodoro():
    global stop_flag
    seconds = pomo_minutes.value * 60
    start = time.time()
    while seconds > 0 and not stop_flag:
        m, s = divmod(seconds, 60)
        status.value = f'{int(m):02d}:{int(s):02d}'
        time.sleep(1)
        seconds -= 1
    status.value = 'Done!' if not stop_flag else 'Stopped'
    logs = load_data('pomodoro.json', {'logs': []})
    logs['logs'].append({'start': start, 'minutes': pomo_minutes.value})
    save_data('pomodoro.json', logs)

def on_start(b):
    global stop_flag
    stop_flag = False
    threading.Thread(target=run_pomodoro).start()

def on_stop(b):
    global stop_flag
    stop_flag = True

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)


## Eisenhower Matrix

In [ ]:
import ipywidgets as widgets
from IPython.display import display

categories = ['Do Now', 'Schedule', 'Delegate', 'Delete']
tasks = load_data('eisenhower.json', {c: [] for c in categories})

task_in = widgets.Text(description='Task:')
cat_dd = widgets.Dropdown(options=categories, description='Category:')
add_btn = widgets.Button(description='Add')
matrix_out = widgets.Output()


def render_matrix():
    matrix_out.clear_output()
    with matrix_out:
        for c in categories:
            print('== ' + c + ' ==')
            for i, t in enumerate(tasks[c], 1):
                print(f'{i}. {t}')
            print()

def add_task(b):
    t = task_in.value.strip()
    if t:
        tasks[cat_dd.value].append(t)
        save_data('eisenhower.json', tasks)
        task_in.value = ''
        render_matrix()

add_btn.on_click(add_task)

display(task_in, cat_dd, add_btn, matrix_out)
render_matrix()


## Day Planner

In [ ]:
import pandas as pd
from IPython.display import display

planner = load_data('planner.json', [])

task_in = widgets.Text(description='Task:')
time_in = widgets.Text(description='Time (HH:MM):')
add_btn = widgets.Button(description='Add')
table_out = widgets.Output()


def render_plan():
    table_out.clear_output()
    with table_out:
        df = pd.DataFrame(planner, columns=['Time', 'Task'])
        display(df)


def add_plan(b):
    t = task_in.value.strip()
    tm = time_in.value.strip()
    if t and tm:
        planner.append([tm, t])
        planner.sort()
        save_data('planner.json', planner)
        task_in.value = ''
        time_in.value = ''
        render_plan()

add_btn.on_click(add_plan)

display(task_in, time_in, add_btn, table_out)
render_plan()


## Task Manager

In [ ]:
tasks_data = load_data('tasks.json', [])

text_task = widgets.Text(description='Task:')
priority = widgets.Dropdown(options=['Low','Medium','High'], value='Medium', description='Priority:')
add_btn = widgets.Button(description='Add')
tasks_out = widgets.Output()


def render_tasks():
    tasks_out.clear_output()
    with tasks_out:
        df = pd.DataFrame(tasks_data, columns=['Task','Priority'])
        display(df)


def add_task(b):
    t = text_task.value.strip()
    if t:
        tasks_data.append([t, priority.value])
        save_data('tasks.json', tasks_data)
        text_task.value = ''
        render_tasks()

add_btn.on_click(add_task)

display(text_task, priority, add_btn, tasks_out)
render_tasks()


## Task Breakdown

In [ ]:
breakdowns = load_data('breakdown.json', {})

main_task_in = widgets.Text(description='Main Task:')
add_main_btn = widgets.Button(description='Add Main Task')
main_select = widgets.Dropdown(options=list(breakdowns.keys()), description='Select:')
subtask_in = widgets.Text(description='Subtask:')
add_sub_btn = widgets.Button(description='Add Subtask')
output = widgets.Output()


def render_breakdown():
    main_select.options = list(breakdowns.keys())
    output.clear_output()
    with output:
        for m, subs in breakdowns.items():
            print(m)
            for s in subs:
                print('  -', s)


def add_main(b):
    m = main_task_in.value.strip()
    if m and m not in breakdowns:
        breakdowns[m] = []
        save_data('breakdown.json', breakdowns)
        main_task_in.value = ''
        render_breakdown()

def add_sub(b):
    m = main_select.value
    s = subtask_in.value.strip()
    if m and s:
        breakdowns[m].append(s)
        save_data('breakdown.json', breakdowns)
        subtask_in.value = ''
        render_breakdown()

add_main_btn.on_click(add_main)
add_sub_btn.on_click(add_sub)

display(main_task_in, add_main_btn, main_select, subtask_in, add_sub_btn, output)
render_breakdown()


## Habit Tracker

In [ ]:
import datetime

habits = load_data('habits.json', {})

habit_in = widgets.Text(description='Habit:')
add_habit_btn = widgets.Button(description='Add Habit')
tracker_out = widgets.Output()


def render_habits():
    tracker_out.clear_output()
    with tracker_out:
        today = datetime.date.today()
        dates = [today + datetime.timedelta(days=i) for i in range(7)]
        header = ['Habit'] + [d.strftime('%m-%d') for d in dates]
        rows = []
        for h, checks in habits.items():
            row = [h]
            for d in dates:
                key = d.isoformat()
                row.append('✓' if checks.get(key) else '')
            rows.append(row)
        df = pd.DataFrame(rows, columns=header)
        display(df)


def add_habit(b):
    h = habit_in.value.strip()
    if h and h not in habits:
        habits[h] = {}
        save_data('habits.json', habits)
        habit_in.value = ''
        render_habits()

add_habit_btn.on_click(add_habit)

display(habit_in, add_habit_btn, tracker_out)
render_habits()


## Routine Tool

In [ ]:
routine = load_data('routine.json', [])

name_in = widgets.Text(description='Step:')
duration_in = widgets.IntText(description='Minutes:', value=1)
add_step_btn = widgets.Button(description='Add Step')
start_btn = widgets.Button(description='Start Routine')
stop_btn = widgets.Button(description='Stop Routine')
output = widgets.Output()


def render_routine():
    output.clear_output()
    with output:
        for step in routine:
            print(step['name'], '-', step['minutes'], 'min')


def add_step(b):
    name = name_in.value.strip()
    mins = duration_in.value
    if name:
        routine.append({'name': name, 'minutes': mins})
        save_data('routine.json', routine)
        name_in.value = ''
        duration_in.value = 1
        render_routine()

running = False

def run_routine():
    global running
    running = True
    for step in routine:
        if not running:
            break
        output.append_stdout('
' + step['name'] + '
')
        for i in range(step['minutes'] * 60, 0, -1):
            if not running:
                break
            output.append_stdout(f"{i//60:02d}:{i%60:02d}")
            time.sleep(1)
        output.append_stdout('
')
    running = False


def start_routine_btn(b):
    if not running:
        threading.Thread(target=run_routine).start()


def stop_routine_btn(b):
    global running
    running = False

add_step_btn.on_click(add_step)
start_btn.on_click(start_routine_btn)
stop_btn.on_click(stop_routine_btn)

display(name_in, duration_in, add_step_btn, start_btn, stop_btn, output)
render_routine()


## Focus Mode

In [ ]:
from IPython.display import Javascript

toggle_btn = widgets.Button(description='Toggle Focus')


def toggle(b):
    display(Javascript('''
    var cells = document.querySelectorAll('.cell.code_cell');
    cells.forEach(function(c){
      c.style.display = (c.style.display === 'none') ? '' : 'none';
    });
    '''))

toggle_btn.on_click(toggle)

display(toggle_btn)


## Rewards

In [ ]:
rewards = load_data('rewards.json', {'points': 0})
points_label = widgets.Label()
inc_btn = widgets.Button(description='Add Point')


def render_points():
    points_label.value = f"Points: {rewards['points']}"


def add_point(b):
    rewards['points'] += 1
    save_data('rewards.json', rewards)
    render_points()

inc_btn.on_click(add_point)

display(inc_btn, points_label)
render_points()


## Calendar Tool

In [ ]:
!pip -q install ics
from ics import Calendar

ics_path = widgets.Text(description='ICS Path:')
load_btn = widgets.Button(description='Load')
cal_out = widgets.Output()


def load_ics(b):
    path = ics_path.value
    if os.path.exists(path):
        with open(path, 'r') as f:
            c = Calendar(f.read())
        cal_out.clear_output()
        with cal_out:
            for e in c.events:
                print(e.begin.humanize(), '-', e.name)

load_btn.on_click(load_ics)

display(ics_path, load_btn, cal_out)


All tools above save data to your Google Drive folder so you can access them later on any device. Enjoy!